# **Author:** Mukhamedali Daniyaruly

# **Topic:** Training Linear layer in Vision Language Models

# **Date:** 22/01/2026


In [ ]:
!pip install -U bitsandbytes transformers

In [ ]:
import os
import math
import random
import glob
import json
from tqdm.auto import tqdm
from google.colab import drive
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import wandb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, AutoProcessor, BitsAndBytesConfig, get_cosine_schedule_with_warmup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# Configuration

In [ ]:
drive.mount("/content/drive")
class Configuration:
  def __init__(self):
    # Paths
    self.data_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/coco_kazakh_train_FULL.json"
    self.images_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/images"
    self.save_path = "/content/drive/MyDrive/Vision KazLM/3types/linear_layer/training_checkpoint"

    # Models
    self.llm_model_name = "issai/LLama-3.1-KazLLM-1.0-8B"
    self.vision_model_name = "google/siglip-so400m-patch14-384"

    # Hyperparameters
    self.vision_embed_dim = 1152
    self.llm_embed_dim = 4096
    self.max_length = 128
    self.epochs = 1
    self.lr = 2e-5
    self.max_length = 128
    self.batch_size = 2
    self.grad_accumulation = 32
    self.log_steps = 2
    self.warmup_ratio = 0.05
    self.weight_decay = 0.01

    # device
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

config = Configuration()
os.makedirs(config.save_path, exist_ok=True)

In [ ]:
wandb.init(
    project_name=config.project_name,
    run_name=config.run_name,
    config={
        "learning_rate": config.lr,
        "batch_size": config.batch_size * config.grad_accumulation,
        "epochs": config.epochs,
        "architecture": "SigLIP + Linear + KazLM",
        "dataset": "COCO-Kaz-113k"
    }
)

In [ ]:
def log_validation_samples(model, processor, tokenizer, val_dataset, num_samples=4):
  model.eval()

  table = wandb.Table(columns=["Image", "True Caption", "Generated Caption"])

  indices = random.sample(len(val_dataset), num_samples)

  for idx in indices:

    entry = val_dataset[idx]

    raw_image_path = os.path.join(config.images_root, val_dataset.data[idx + len(val_dataset.train)]["image_path"])
    raw_image = Image.open(raw_image_path).convert("RGB")
    true_caption = random.choice(val_dataset.data[idx + len(val_dataset.train_data)]["captions_kk"])

    pixel_values = processor(images=[raw_image], return_tensors='pt').pixel_values.to(config.device)

    table.add_data(wand.Image(raw_image), true_caption, "Not Generated Yet")

  wandb.log({
      "Validation samples": table
  })

# Dataset

In [ ]:
class VLMDataset(Dataset):
    def __init__(self, json_path, images_root, tokenizer, processor, max_length=128):
        self.images_root = images_root
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length

        print(f"Loading data from {json_path}...")
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        print(f"Loaded {len(self.data)} samples")

    def __len__(self):

      # Return Length of dataset
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]

        # 1. Load Image
        image_path = os.path.join(self.images_root, entry['image_path'])
        try:
            image = Image.open(image_path).convert('RGB')
        except:
            return self.__getitem__(random.randint(0, len(self.data)-1)) # Fallback

        # 2. Process Image (SigLIP)
        # pixel_values: [3, 384, 384]
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        # 3. Process Text
        # Instruction: "Суреттегі көрініс: [CAPTION]"
        caption = random.choice(entry['captions_kk'])
        prompt = "Суреттегі көрініс: "
        full_text = prompt + caption + self.tokenizer.eos_token

        # Tokenize
        tokenized = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = tokenized.input_ids.squeeze(0)
        attention_mask = tokenized.attention_mask.squeeze(0)

        # 4. Create Labels (Masking user prompt)
        # We want to model train predict caption, not instruction
        labels = input_ids.clone()

        # Находим длину промпта в токенах, чтобы скрыть его
        prompt_len = len(self.tokenizer(prompt).input_ids) - 1 # -1 for deleting bos token
        labels[:prompt_len] = -100 # Mask prompt
        labels[labels == self.tokenizer.pad_token_id] = -100 # Mask padding

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
config.data_path

In [ ]:
with open(config.data_path, "r", encoding="utf-8") as file:
  data = json.load(file)

data, len(data)

In [ ]:
images_path = os.path.join("coco_karpathy", "images")
images_path

In [ ]:
os.listdir("coco_karpathy")

In [ ]:
annot_path = os.path.join("coco_karpathy", "annotations")
os.listdir(annot_path)

In [ ]:
train_images_path = os.path.join("coco_karpathy", "images/train2014")
val_images_path = os.path.join("coco_karpathy", "images/val2014")
train_images_path, val_images_path


In [ ]:
tokenizer.eos_token

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
tokenizer.pad_token_id

In [ ]:
full_dataset = VLMDataset(
    json_path = config.data_path,
    images_root = images_path,
    tokenizer = tokenizer,
    processor = processor,
    max_length = 128
)

In [ ]:
full_dataset[0]

In [ ]:
def collate_fn(batch):
  px, ins, att, lb = batch
  px = torch.stack(px)
  ins = torch.stack(ins)
  att = torch.stack(att)
  lb = torch.stack(lb)

  return px, ins, att, lb

In [ ]:
# initialize train, val and test dataset
train_dataset, val_dataset, test_dataset =

train_loader, val_loader, test_loader = DataLoader(train_dataset, shuffle=True, batch_size=config.batch_size), DataLoader(val_dataset, shuffle=False, batch_size=config.batch_size), DataLoader(test_dataset, shuffle=False, batch_size=config.batch_size)

In [ ]:
sample_batch = DataLoader(full_dataset, shuffle=True, batch_size=1)

In [ ]:
for batch in sample_batch:
  print(batch)
  break

# Architecture

In [ ]:
class VisionEncoder(nn.Module):
  def __init__(self, vision_encoder):
    super.__init__()

    self.vision_encoder = vision_encoder

    for p in self.vision_encoder.parameters():
      p.requires_grad=False

  def forward(self, pixel_values):
    outs = self.vision_encoder.vision_model(pixel_values=pixel_values)
    return outs["last_hidden_state"]

In [ ]:
class KazVLM(nn.Module):
  def __init__(self, vision_model, llm_model, config):
      super().__init__()
      self.vision_encoder = vision_model
      self.llm = llm_model

      # 1. Projection Layer (Trainable)
      # In this layer we convert 1152 to 4096
      self.projector = nn.Linear(config.vision_embed_dim, config.llm_embed_dim)

      # Freeze Vision & LLM
      for param in self.vision_encoder.parameters():
          param.requires_grad = False
      for param in self.llm.parameters():
          param.requires_grad = False

      print(f"Model Initialized. Trainable Params: {sum(p.numel() for p in self.projector.parameters())}")

  def forward(self, pixel_values, input_ids, attention_mask, labels=None):
       # 1. Get Image Embeddings [Batch, 3, 384, 384] -> [Batch, 729, 1152]
      with torch.no_grad():
          vision_outputs = self.vision_encoder.vision_model(pixel_values)
          image_features = vision_outputs.last_hidden_state

      # 2. Project Image Features -> [Batch, 729, 4096]
      image_embeds = self.projector(image_features)

      # 3. Get Text Embeddings -> [Batch, Seq_Len, 4096]
      text_embeds = self.llm.get_input_embeddings()(input_ids)

      # 4. Concatenate: [Image_Embeds, Text_Embeddings]
      # Model looks at the image then loot at the text
      inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1)

      # 5. Adjust Attention Mask
      # We create mask for 729 image tokens
      batch_size = input_ids.shape[0]
      image_mask = torch.ones((batch_size, image_embeds.shape[1]), device=image_embeds.device)
      full_attention_mask = torch.cat([image_mask, attention_mask], dim=1)

      # 6. Adjust Labels (Если тренировка)
      if labels is not None:
          # We do not predict image tokens
          image_labels = torch.full((batch_size, image_embeds.shape[1]), -100, device=labels.device)
          full_labels = torch.cat([image_labels, labels], dim=1)
      else:
          full_labels = None

      # 7. Pass to LLM
      outputs = self.llm(
          inputs_embeds=inputs_embeds,
          attention_mask=full_attention_mask,
          labels=full_labels
      )

      return outputs

# Download SigLIP and KazLM model from huggingface

In [ ]:
# We need to verify us to get access for using KazLM model
from huggingface_hub import login
login("YOUR_HUGGINGFACE_TOKEN")

In [ ]:
# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
# Download tokenizer and set padding token
tokenizer = AutoTokenizer.from_pretrained(config.llm_model_name)
tokenizer_pad_token = tokenizer.eos_token

# Download processor
processor = AutoProcessor.from_pretrained(config.vision_model_name)

In [ ]:
# Download SiLIP model and set to cuda
vision_model = AutoModel.from_pretrained(
    config.vision_model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

In [ ]:
# Download KazLM and Quantize it
llm_model = AutoModelForCausalLM.from_pretrained(config.llm_model_name, quantization_config=bnb_config, device_map="cuda")

In [ ]:
llm_model

# Training configuration

In [ ]:
# Initialize a Vision Language Model
model = KazVLM(vision_model, llm_model, config)

In [ ]:
model = model.to(device)

In [ ]:
# Dataset
dataset = VLMDataset(config.data_path, config.image_root, tokenizer, processor)
train_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)

In [ ]:
# optimizer
optimizer = torch.optim.AdamW(model.projector.parameters(), lr=config.lr)
num_training_steps = int((1 * config.epochs) // config.grad_accumulation)
num_warmup_steps = int(num_training_steps * config.warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_training_steps=num_training_steps, num_warmup_steps=num_warmup_steps
)
scaler = torch.amp.GradScaler()

In [ ]:
for batch in full_dataset:
  px = batch["pixel_values"]
  ins = batch["input_ids"]
  att = batch["attention_mask"]
  lab = batch["labels"]

  px = px.unsqueeze(0)
  ins = ins.unsqueeze(0)
  att = att.unsqueeze(0)
  lab = lab.unsqueeze(0)
  print(px.shape, ins.shape, att.shape, lab.shape)
  break

In [ ]:
px.shape, px.dtype

In [ ]:
outs = vision_model.vision_model(pixel_values=px)
outs

In [ ]:
for batch in full_dataset:
  px, ins, att, lab = batch["pixel_values"], batch["input_ids"], batch["attention_mask"], batch["labels"]
  px, ins, att, lab = px.unsqueeze(0), ins.unsqueeze(0), att.unsqueeze(0), lab.unsqueeze(0)

  # Only pixel_values should be bfloat16, other tensors should retain their original integer type (long)
  px = px.to(device, dtype=torch.bfloat16)
  ins = ins.to(device) # Keep as long
  att = att.to(device) # Keep as long
  lab = lab.to(device) # Keep as long

  with torch.no_grad():
    with torch.amp.autocast(dtype=torch.bfloat16, device_type=device):

      outs = model(px, ins, att, lab)
  break

outs

In [ ]:
# ⚡️ Быстрый тест: учится ли модель вообще?
print("🧪 Sanity Check: Обучение на одном батче...")

# Временный оптимизатор (только для теста)
test_optimizer = torch.optim.AdamW(model.projector.parameters(), lr=1e-4)

# Прогоним этот же батч 10 раз
for i in range(10):
    # 1. Forward
    with torch.amp.autocast(dtype=torch.bfloat16, device_type=device):
        outputs = model(px, ins, att, lab)
        loss = outputs.loss

    # 2. Backward
    loss.backward()
    test_optimizer.step()
    test_optimizer.zero_grad()

    print(f"Step {i+1}: Loss = {loss.item():.4f}")

print("✅ Если лосс падает — ты готов к большой тренировке!")

In [ ]:
outs.loss, outs.

In [ ]:
for epoch in range(config.epochs):

  # set to training
  model.train()

  pbar = tqdm(train_loader)
  total_loss = 0

  # iterate train loader
  for step, batch in enumerate(pbar):
    pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Use mixed precision
    with torch.amp.autocast(device=device, dtype=torch.bfloat16):
      outputs = model(pixel_values, input_ids, attention_mask, labels)
      loss = outputs.loss
      loss = loss / config.grad_accumulation

    scaler.scale(loss).backward()

    # Use gradient accumulation
    if (step + 1) % config.grad_accumulation == 0:
      scaler.unscale_(optimizer)
      torch.nn.init.clip_grad_norm_(model.projection.parameters(), 1.0)

      scaler.update(optimizer)
      scaler.update()
      scheduler.update()
      optimizer.zero_grad()

    total_loss += loss.item()
    pbar.set_description(f"Epoch: {epoch + 1} | Loss: {loss.item() * config.grad_accumulation:.4f} | PPL: {math.exp(loss.item() * config.grad_accumulation):.4f}")

  # Logging on validation dataset
  if (epoch + 1) % config.log_step == 0:

    # set to evaluation state in validation part
    model.eval()

    val_total_loss = 0

    with torch.no_grad():

    # iterate val loader
      for batch in tqdm(val_loader, desc="Validation"):

        pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch = batch["labels"].to(device)

        # Use mixed precision
        with torch.amp.autocast(device=device, dtype=torch.bfloat16):
          outputs = model(pixel_values, input_ids, attention_mask, labels)
          val_loss = outputs.loss

        # compute validation loss
        val_total_loss += val_loss.item()

      avg_val_loss = val_total_loss / len(val_loader)
      avg_val_ppl = math.exp(avg_val_loss)

      print(f"Epoch: {epoch + 1} | Val_Loss: {avg_val_loss} | Val_PPL: {avg_val_ppl}")

  # Compute training loss
  avg_loss = total_loss / len(train_loader)
  avg_ppl = math.exp(avg_loss)
  print("-----" * 20)
  print(f"Epoch: {epoch + 1} | Loss: {avg_loss} | PPL: {avg_ppl}")

  # Save each epoch
  torch.save({
      "epoch": {epoch + 1},
      "model_projection_state_dict": model.projector.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "scaler_state_dict": scaler.state_dict(),
      "loss": avg_loss,
      "ppl": avg_ppl,
      "val_loss": avg_val_loss,
      "val_ppl": avg_val_ppl
  }, f"{confing.save_path}/projector_epoch_{epoch+1}.pt")

# Evaluate

In [ ]:
model.eval()

losses = 0

for batch in test_loader:
  with torch.no_grad():
    pixel_values, input_ids, attention_mask, labels = [b.to(device) for batch]

    with torch.amp.autocast(torch.bfloat16):
      outs = model(pixel_values, input_ids, attention_mask, labels)

    loss += outs.loss.item()

print(f"Loss: {loss}, ppl: {math.exp(loss)}")

# Inference

In [ ]:
def inference_check(image, prompt="Суретте не көріп тұрсың?"):
  try:
    image = Image.open(image).convert("RGB")
  except Exception as e:
    print(f"Error at loading image {image}: {e}")

  pixel_values = processor(images=[image], return_tensors="pt").squeeze(0)
  image_features = vision_model(**pixel_values)
  image_features = model.linear_projection(image_features)

  text_inputs = tokenizer(
      prompt,
      max_length=128,
      truncation=True,
      padding="max_length",
      return_tensors="pt"
  )

  input_ids = text_inputs.input_ids
  attention_mask = text_inputs.attention_mask

  labels = input_ids.clone()
  labels[labels == tokenizer.pad_token_id] = -100

  outs = model(pixel_values, text=input_ids, attention_mask=attention_mask, labels=labels)

  return tokenizer.decode(outs[0][outs["input_ids"].shape[-1]:])

# Data sanity check

In [ ]:
from PIL import Image

In [ ]:
import os
import json
from google.colab import drive

drive.mount("/content/drive")
data_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/coco_kazakh_train_FULL.json"

with open(data_path, "r", encoding="utf-8") as file:
  data = json.load(file)

data

In [ ]:
import os

# 1. Создаем структуру
root_dir = "/content/coco_karpathy"
os.makedirs(f"{root_dir}/images", exist_ok=True)
os.makedirs(f"{root_dir}/annotations", exist_ok=True)

# 2. Скачиваем Картинки 2014 (Train + Val) ~13GB + 6GB
# Важно: В Karpathy split картинки из train2014 и val2014 смешиваются
!wget -c http://images.cocodataset.org/zips/train2014.zip -O {root_dir}/train2014.zip
!unzip -q {root_dir}/train2014.zip -d {root_dir}/images/
!rm {root_dir}/train2014.zip

!wget -c http://images.cocodataset.org/zips/val2014.zip -O {root_dir}/val2014.zip
!unzip -q {root_dir}/val2014.zip -d {root_dir}/images/
!rm {root_dir}/val2014.zip

In [ ]:
main_path = "/content/coco_karpathy"
os.listdir(main_path)

In [ ]:
images_path = os.path.join(main_path, "images")
os.listdir(images_path)

In [ ]:
train_images = os.path.join(images_path, "train2014")
val_images = os.path.join(images_path, "val2014")
os.listdir(val_images)

In [ ]:
image_path = os.path.join(val_images, 'COCO_val2014_000000230360.jpg')
os.path.exists(image_path)

In [ ]:
Image.open(image_path).convert("RGB")

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
len(data)

In [ ]:
data[0]